## Modelowanie układów przepływowych — kilwater łodzi płynącej w płytkiej wodzie

Ruchome zaburzenie batymetrii w równaniach płytkiej wody, rozwiązane w PyMPDATA.

### Pytanie

Pod jakim kątem rozchodzi się kilwater za łódką płynącą ze stałą prędkością $U$ po wodzie o głębokości $H$?

Dla fal długich relacja dyspersji jest liniowa, $\omega = ck$ z $c=\sqrt{gH}$, więc zaburzenie mieści się w klinie o połowie kąta

$$\sin\theta = \frac{c}{U} = \frac{1}{\mathrm{Fr}}, \qquad \mathrm{Fr} = \frac{U}{\sqrt{gH}}$$

In [ ]:
import sys

if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA')

In [ ]:
import numpy as np
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions

### Model

Kadłub jest ruchomym wypiętrzeniem dna $b(x-Ut,\,y)$ opisanym gładkim prostokątem (supergauss); w równaniach płytkiej wody wchodzi wyłącznie przez gradient powierzchni $\zeta = h - b$.

Względem solwera z wykładu zmienione są dwie rzeczy: batymetria zależy od czasu oraz różnica centralna w członie ciśnienia jest periodyczna (`np.roll`), tak jak warunek brzegowy. Solwer zwraca $\zeta$ zamiast pełnych pól.

In [ ]:
G = 9.81
DEPTH = 1.0
C0 = np.sqrt(G * DEPTH)

NX, NY, DX = 600, 300, 1.0
DT, FR, AMPLITUDE = 0.16, 2.0, 0.03
LENGTH, WIDTH = 20.0, 7.5
X_START, X_STOP = 0.25 * NX * DX, 0.75 * NX * DX

In [ ]:
def hull_profile(*, X, Y, x0, y0, length, width, amplitude):
    return amplitude * np.exp(-(((X - x0) / length) ** 4 + ((Y - y0) / width) ** 4))


def periodic_gradient(field, axis):
    return 0.5 * (np.roll(field, -1, axis) - np.roll(field, 1, axis))

In [ ]:
class ShallowWater:
    def __init__(self, *, n, dx=1.0, depth=DEPTH, hull=None, options=None):
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        self.dx = dx
        self.depth = depth
        self.hull = hull
        self.X, self.Y = np.meshgrid(
            np.arange(n[0]) * dx, np.arange(n[1]) * dx, indexing='ij'
        )
        self.y0 = 0.5 * (n[1] - 1) * dx
        self.bathymetry = np.full(n, depth)
        if hull is not None:
            self.set_hull(**hull(0.0))
        stepper = Stepper(options=options, grid=n)
        kwargs = {
            'boundary_conditions': [boundary_conditions.Periodic()] * len(n),
            'halo': options.n_halo,
        }
        advectees = {
            'h': ScalarField(self.bathymetry.copy(), **kwargs),
            'uh': ScalarField(np.zeros(n), **kwargs),
            'vh': ScalarField(np.zeros(n), **kwargs),
        }
        self.advector = VectorField(
            (
                np.zeros((n[0] + 1, n[1])),
                np.zeros((n[0], n[1] + 1)),
            ),
            **kwargs,
        )
        self.solvers = {
            k: Solver(stepper, v, self.advector) for k, v in advectees.items()
        }

    def __getitem__(self, key):
        return self.solvers[key].advectee.get()

    def set_hull(self, **kwargs):
        self.bathymetry[:] = self.depth + hull_profile(
            X=self.X, Y=self.Y, y0=self.y0, **kwargs
        )

    def _apply_half_rhs(self, *, key, axis, g_times_dt_over_dxy):
        self[key][:] -= (
            0.5
            * g_times_dt_over_dxy
            * self['h']
            * periodic_gradient(self['h'] - self.bathymetry, axis)
        )

    def _update_courant_numbers(self, *, axis, key, mask, dt_over_dxy):
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)

        all = slice(None, None)
        all_but_last = slice(None, -1)
        all_but_first_and_last = slice(1, -1)

        velocity_at_cell_boundaries = (
            velocity[
                (
                    (all_but_last, all),
                    (all, all_but_last),
                )[axis]
            ]
            + np.diff(velocity, axis=axis) / 2
        )
        courant_number = self.advector.get_component(axis)[
            (
                (all_but_first_and_last, all),
                (all, all_but_first_and_last),
            )[axis]
        ]
        courant_number[:] = velocity_at_cell_boundaries * dt_over_dxy[axis]
        assert np.amax(np.abs(courant_number)) <= 1

    def __call__(self, *, nt, g, dt_over_dxy, outfreq=1, eps=1e-7):
        output = {'eta': [], 'x0': [], 'mass': []}
        dt = dt_over_dxy[0] * self.dx
        for it in range(nt + 1):
            if it != 0:
                if self.hull is not None:
                    self.set_hull(**self.hull(it * dt))
                mask = self['h'] > eps
                for axis, key in enumerate(('uh', 'vh')):
                    self._update_courant_numbers(
                        axis=axis, key=key, mask=mask, dt_over_dxy=dt_over_dxy
                    )
                self.solvers['h'].advance(n_steps=1)
                for axis, key in enumerate(('uh', 'vh')):
                    self._apply_half_rhs(
                        key=key,
                        axis=axis,
                        g_times_dt_over_dxy=g * dt_over_dxy[axis],
                    )
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(
                        key=key,
                        axis=axis,
                        g_times_dt_over_dxy=g * dt_over_dxy[axis],
                    )
            if it % outfreq == 0:
                output['eta'].append(self['h'] - self.bathymetry)
                output['mass'].append(self['h'].sum())
                output['x0'].append(0.0 if self.hull is None else self.hull(it * dt)['x0'])
        return output

### Symulacja

Kadłub startuje w $x = 0.25L_x$ i płynie do $0.75L_x$; w chwili $t$ jego położenie to $x_0 + Ut$.

In [ ]:
SPEED = FR * C0
NT = int(round((X_STOP - X_START) / (SPEED * DT)))
FRAME_EVERY = 4


def hull(t):
    return dict(x0=X_START + SPEED * t, length=LENGTH, width=WIDTH, amplitude=AMPLITUDE)


model = ShallowWater(n=(NX, NY), dx=DX, hull=hull)
output = model(nt=NT, g=G, dt_over_dxy=(DT / DX, DT / DX), outfreq=FRAME_EVERY)
eta = output['eta']
boat_x = np.array(output['x0'])
mass = np.array(output['mass'])

assert np.isfinite(eta[-1]).all()
assert abs(mass[-1] / mass[0] - 1.0) < 1e-12
assert np.abs(eta[-1]).max() < 0.2 * DEPTH

print(f'{NX}x{NY}, dx={DX} m, dt={DT} s, CFL={C0 * DT / DX:.2f}, frames={len(eta)}')
print(f'Fr={FR:.1f}, U={SPEED:.2f} m/s, nt={NT}, t={NT * DT:.1f} s')
print(f'mass drift={mass[-1] / mass[0] - 1:+.1e}, max|eta|={np.abs(eta[-1]).max():.3f} m')

### Wizualizacja

Kolor to $\zeta$, linia przerywana to przewidywany klin $\pm\theta$ o wierzchołku w łódce. Duże blade koło to fala startowa rozchodząca się od miejsca, w którym łódka ruszyła; warunki periodyczne sprawiają, że czoło fali opuszczające domenę wraca od drugiej strony — stąd słabe struktury przy $x=0$ i $x=600$ m.

In [ ]:
theta = np.arcsin(1 / FR)
STEP = 4
x = np.arange(NX) * DX
y = np.arange(NY) * DX
y0 = 0.5 * NY * DX


def plot(i, *, step=STEP):
    boat = boat_x[i]
    fig, ax = pyplot.subplots(figsize=(7.0, 3.6), constrained_layout=True)
    mesh = ax.pcolormesh(
        x[::step],
        y[::step],
        eta[i][::step, ::step].T,
        cmap='RdBu_r',
        vmin=-AMPLITUDE,
        vmax=AMPLITUDE,
    )
    ax.plot(
        boat + 0.5 * LENGTH * np.array([-1, -1, 1, 1, -1]),
        y0 + 0.5 * WIDTH * np.array([-1, 1, 1, -1, -1]),
        'k-',
        lw=1.0,
    )
    for sign in (+1, -1):
        ax.plot([0, boat], y0 + sign * np.tan(theta) * boat * np.array([1, 0]), 'k--', lw=0.8)
    ax.set_xlim(0, NX * DX)
    ax.set_ylim(0, NY * DX)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_title(
        rf'$t = {i * FRAME_EVERY * DT:.0f}$ s,  $\mathrm{{Fr}} = {FR:.1f}$,  '
        rf'$\theta = {np.degrees(theta):.1f}^\circ$'
    )
    pyplot.colorbar(mesh, ax=ax, shrink=0.85, label=r'$\zeta$ [m]').solids.set_rasterized(False)
    return fig


plot(len(eta) - 1)
show_plot('wake')

### Animacja

Kamera obejmuje całą domenę, więc widać narastanie klina: łódka pokonuje 300 m w czasie $\approx 48$ s. Klatki są gęstsze niż punkty pomiarowe wykresu, dlatego animacja używa gęstszej siatki ($\Delta x = 2$ m zamiast $4$ m).

In [ ]:
show_anim(lambda i: plot(i, step=2), range(len(eta)), duration=0.05, gif_file='wake.gif')

### Pomiar kąta

Okno pomiarowe wycina falę startową i sam kadłub: $x\in[x_0+ct+\Delta,\,x_0+Ut-\Delta]$. W każdej kolumnie bierzemy najdalszy $y$ o $|\zeta|>0.2A$ i dopasowujemy $|y-y_0|=\tan\theta\,(x_{boat}-x)+b$ z wolnym wyrazem $b$: kadłub ma skończoną długość, więc efektywne źródło leży ok. 30 m przed jego środkiem. Dopasowanie wymuszone przez środek daje 37° zamiast 30°.


In [ ]:
def run_case(fr, *, dx=1.0, dt=0.16, travel=300.0, x_start=150.0, nx=600, ny=300):
    u = fr * C0
    n = (int(round(nx / dx)), int(round(ny / dx)))
    nt = int(round(travel / (u * dt)))
    outfreq = max(1, int(round(4.0 / dx)))

    def hull(t):
        return dict(x0=x_start + u * t, length=LENGTH, width=WIDTH, amplitude=AMPLITUDE)

    model = ShallowWater(n=n, dx=dx, hull=hull)
    out = model(nt=nt, g=G, dt_over_dxy=(dt / dx, dt / dx), outfreq=outfreq)
    eta_last = np.asarray(out['eta'][-1]).copy()
    boat_x = np.asarray(out['x0'])
    mass = np.asarray(out['mass'])
    shift = int(round((boat_x[-1] - boat_x[-2]) / dx))
    diff = eta_last[shift:] - np.asarray(out['eta'][-2])[:eta_last.shape[0] - shift]
    a = int((boat_x[-1] - travel * 0.8) / dx)
    b = int((boat_x[-1] - 40) / dx)
    rms = float(np.sqrt(np.mean(diff[a:b] ** 2)))
    qa = int((boat_x[-1] + 2 * LENGTH) / dx)
    qb = int((boat_x[-1] + 100) / dx)
    front = (x_start - C0 * nt * dt) % (n[0] * dx)
    if front > boat_x[-1]:
        qb = min(qb, int((front - 30) / dx))
    quiet = float(np.abs(eta_last[qa:qb]).max()) if qb > qa else None
    return dict(
        fr=fr, dx=dx, dt=dt, u=u, t=nt * dt, n=n, boat_x=float(boat_x[-1]),
        x0=x_start, eta=eta_last, mass_drift=float(mass[-1] / mass[0] - 1.0),
        rms=rms, quiet=quiet,
    )


def wake_angle(case, *, threshold=0.2, margin=5.0):
    eta = case['eta']
    nx, ny = eta.shape
    dx = case['dx']
    x = np.arange(nx) * dx
    y = np.arange(ny) * dx
    y0 = 0.5 * (ny - 1) * dx
    lo = case['x0'] + C0 * case['t'] + margin
    hi = case['boat_x'] - margin
    cols = np.where((x >= lo) & (x <= hi))[0]
    pts = []
    for j in cols:
        idx = np.where(np.abs(eta[j]) > threshold * AMPLITUDE)[0]
        if len(idx):
            pts.append((case['boat_x'] - x[j], float(np.max(np.abs(y[idx] - y0)))))
    pts = np.asarray(pts)
    design = np.column_stack((pts[:, 0], np.ones(len(pts))))
    slope, intercept = np.linalg.lstsq(design, pts[:, 1], rcond=None)[0]
    theta = float(np.degrees(np.arctan(slope)))
    theory = float(np.degrees(np.arcsin(1 / case['fr'])))
    return dict(theta=theta, theory=theory, error=theta - theory,
                intercept=float(intercept), npts=len(pts), window=(float(lo), float(hi)))


### Skan liczby Froude’a

Dla każdego $Fr$ łódka pokonuje te same 300 m. Linie przerywane to teoria,  Zielony pasek to okno dopasowania. Wykres poniżej zestawia $\theta$ z $\arcsin(1/Fr)$.

In [ ]:
scan = [1.2, 1.5, 2.0, 3.0]
results = []
for fr in scan:
    case = run_case(fr)
    res = wake_angle(case)
    assert abs(case['mass_drift']) < 1e-12
    assert case['rms'] < 0.05 * AMPLITUDE
    if case['quiet'] is not None:
        assert case['quiet'] < 0.01 * AMPLITUDE
    results.append((case, res))
    quiet_txt = 'n/a' if case['quiet'] is None else f"{case['quiet']:.1e}"
    print(f'Fr={fr:.1f}  pomiar={res["theta"]:.2f}°  teoria={res["theory"]:.2f}°  '
          f'b={res["intercept"]:.1f} m  n={res["npts"]}  rms={case["rms"]:.5f}  '
          f'przód={quiet_txt}')

fig, axs = pyplot.subplots(2, 2, figsize=(11, 5.6), constrained_layout=True)
for ax, (case, res) in zip(axs.ravel(), results):
    eta = case['eta']
    dx = case['dx']
    xg = np.arange(eta.shape[0]) * dx
    yg = np.arange(eta.shape[1]) * dx
    step = max(1, int(round(3 / dx)))
    mesh = ax.pcolormesh(xg[::step], yg[::step], eta[::step, ::step].T,
                         cmap='RdBu_r', vmin=-AMPLITUDE, vmax=AMPLITUDE)
    yc = 0.5 * eta.shape[1] * dx
    th = np.radians(res['theory'])
    for sign in (+1, -1):
        ax.plot([case['x0'], case['boat_x']],
                [yc, yc + sign * np.tan(th) * (case['boat_x'] - case['x0'])],
                'k--', lw=0.8)
    lo, hi = res['window']
    ax.axvspan(lo, hi, facecolor='none', edgecolor='#00a000', ls=':', lw=1.2)
    ax.plot(case['boat_x'], yc, 'kv', ms=5)
    ax.set(xlim=(0, case['n'][0] * dx), ylim=(0, case['n'][1] * dx), aspect='equal',
           title=f'Fr={case["fr"]:.1f}: teoria {res["theory"]:.1f}°, pomiar {res["theta"]:.1f}°')
    ax.set_xlabel('$x$ [m]')
    ax.set_ylabel('$y$ [m]')
pyplot.colorbar(mesh, ax=axs, shrink=0.8, label=r'$\zeta$ [m]').solids.set_rasterized(False)
pyplot.savefig('wake_scan.pdf', bbox_inches='tight')
pyplot.savefig('wake_scan.svg', bbox_inches='tight')

fig, ax = pyplot.subplots(figsize=(5.2, 3.2), constrained_layout=True)
fr_line = np.linspace(1.05, 3.4, 200)
ax.plot(fr_line, np.degrees(np.arcsin(1 / fr_line)), 'k-', lw=1.2, label=r'$\arcsin(1/Fr)$')
ax.plot([c['fr'] for c, _ in results], [r['theta'] for _, r in results],
        'o', color='#d2232a', ms=6, label='pomiar')
ax.set(xlim=(1, 3.5), ylim=(15, 70), xlabel=r'$Fr$', ylabel=r'$\theta$ [deg]')
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=8)
pyplot.savefig('angle_vs_fr.pdf', bbox_inches='tight')
pyplot.savefig('angle_vs_fr.svg', bbox_inches='tight')


### Zbieżność przestrzenna i czasowa

Błąd kątu maleje przy dzieleniu $\Delta x$ (ze stałą liczbą Couranta) oraz przy dzieleniu $\Delta t$ (na stałej siatce).

In [ ]:
spatial = []
for dx in (3.0, 2.0, 1.0):
    case = run_case(2.0, dx=dx, dt=0.16 * dx)
    res = wake_angle(case)
    spatial.append((dx, res['error']))
    print(f'dx={dx:.0f}  dt={case["dt"]:.2f}  theta={res["theta"]:.2f}°  '
          f'błąd={res["error"]:+.2f}°  rms={case["rms"]:.5f}')

temporal = []
for dt in (0.32, 0.16, 0.08):
    case = run_case(2.0, dt=dt)
    res = wake_angle(case)
    temporal.append((dt, res['error']))
    print(f'dt={dt:.2f}  theta={res["theta"]:.2f}°  błąd={res["error"]:+.2f}°')

fig, axs = pyplot.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for ax, data, label in ((axs[0], spatial, r'$\Delta x$ [m]'),
                        (axs[1], temporal, r'$\Delta t$ [s]')):
    h = np.array([row[0] for row in data])
    err = np.abs(np.array([row[1] for row in data]))
    ax.loglog(h, err, 'o-', color='#58a756', lw=1.2)
    ax.invert_xaxis()
    ax.grid(alpha=0.3, which='both')
    ax.set_xlabel(label)
    ax.set_ylabel(r'$|\theta_{pom}-\theta_{teor}|$ [deg]')
pyplot.savefig('convergence.pdf', bbox_inches='tight')
pyplot.savefig('convergence.svg', bbox_inches='tight')
